# Qasper Dataset Loader

Notebook này chạy độc lập trên Kaggle/Colab. Không cần clone GitHub, không cần import code từ repo. Mục tiêu hiện tại: tải và kiểm tra bộ dữ liệu Qasper trước khi xây baseline RAG.

In [ ]:
!pip install -q datasets pyarrow

In [ ]:
from datasets import load_dataset

QASPER_REVISION = "cc58ffb39db7ff6ce1951e28e029996bf499304e"
QASPER_BASE_URL = f"https://huggingface.co/datasets/allenai/qasper/resolve/{QASPER_REVISION}/qasper"
QASPER_PARQUET_FILES = {
    "train": f"{QASPER_BASE_URL}/qasper-train.parquet",
    "validation": f"{QASPER_BASE_URL}/qasper-validation.parquet",
    "test": f"{QASPER_BASE_URL}/qasper-test.parquet",
}

ds = load_dataset("parquet", data_files=QASPER_PARQUET_FILES)
ds

In [ ]:
print(ds.keys())
print(ds["train"])
print(ds["validation"])
print(ds["test"])

In [ ]:
sample = ds["validation"][0]

print("ID:", sample["id"])
print("Title:", sample["title"])
print("Abstract preview:", sample["abstract"][:500])
print("Number of QA pairs:", len(sample["qas"]["question"]))
print("Full text sections:", len(sample["full_text"]["section_name"]))

In [ ]:
def iter_answer_records(answers):
    if isinstance(answers, list):
        return answers
    if not isinstance(answers, dict):
        return []

    answer_values = answers.get("answer", [])
    annotation_ids = answers.get("annotation_id", [])
    worker_ids = answers.get("worker_id", [])

    if isinstance(answer_values, dict):
        answer_values = [answer_values]

    records = []
    for idx, answer_value in enumerate(answer_values):
        row = {"answer": answer_value}
        if isinstance(annotation_ids, list) and idx < len(annotation_ids):
            row["annotation_id"] = annotation_ids[idx]
        if isinstance(worker_ids, list) and idx < len(worker_ids):
            row["worker_id"] = worker_ids[idx]
        records.append(row)
    return records


def show_qa(record, index=0):
    question = record["qas"]["question"][index]
    answers = iter_answer_records(record["qas"]["answers"][index])

    print("Question:", question)
    print("Answers:")
    for answer in answers:
        print(answer)

show_qa(sample, 0)

In [ ]:
class QasperRecordView:
    def __init__(self, record):
        self.record = record

    @property
    def title(self):
        return self.record["title"]

    def document_text(self):
        parts = [self.record.get("abstract", "")]
        full_text = self.record.get("full_text", {})
        sections = full_text.get("section_name", [])
        paragraphs_by_section = full_text.get("paragraphs", [])

        for section, paragraphs in zip(sections, paragraphs_by_section):
            parts.append(f"\n## {section}\n")
            parts.extend(str(paragraph) for paragraph in paragraphs)
        return "\n".join(parts)

    def qa_pairs(self):
        qas = self.record["qas"]
        rows = []
        for question_id, question, answers in zip(
            qas["question_id"], qas["question"], qas["answers"]
        ):
            rows.append({
                "question_id": question_id,
                "question": question,
                "answers": iter_answer_records(answers),
            })
        return rows

view = QasperRecordView(sample)
print(view.title)
print(view.document_text()[:1000])
print("QA pairs:", len(view.qa_pairs()))

In [ ]:
import json
from pathlib import Path

output_path = Path("qasper_validation_preview.jsonl")

with output_path.open("w", encoding="utf-8") as f:
    for record in ds["validation"].select(range(5)):
        view = QasperRecordView(record)
        row = {
            "id": record["id"],
            "title": record["title"],
            "document_preview": view.document_text()[:2000],
            "qa_pairs": view.qa_pairs(),
        }
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:", output_path)

## Base RAG Training Runner

Phần dưới là bản self-contained cho Kaggle. Không cần clone/import repo. Logic giống baseline `.py`: chunk tài liệu, build dense index theo từng paper, retrieve top-k, generate bằng small LLM, rồi lưu prediction và summary metrics.

In [ ]:
!pip install -q sentence-transformers transformers torch tqdm numpy

In [ ]:
import json
import re
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
@dataclass(frozen=True)
class Chunk:
    chunk_id: str
    doc_id: str
    title: str
    section: str
    text: str


@dataclass(frozen=True)
class QAExample:
    doc_id: str
    question_id: str
    title: str
    question: str
    gold_answers: list
    evidence: list


def chunk_words(text, chunk_size=180, overlap=40):
    words = text.split()
    if not words:
        return []
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        window = words[start:start + chunk_size]
        if window:
            chunks.append(' '.join(window))
        if start + chunk_size >= len(words):
            break
    return chunks


def build_document_chunks(record, chunk_size=180, overlap=40):
    chunks = []
    chunk_index = 0
    for text in chunk_words(record.get('abstract', ''), chunk_size, overlap):
        chunks.append(Chunk(f"{record['id']}::abstract::{chunk_index}", record['id'], record.get('title', ''), 'abstract', text))
        chunk_index += 1

    full_text = record.get('full_text', {})
    for section, paragraphs in zip(full_text.get('section_name', []), full_text.get('paragraphs', [])):
        section_text = ' '.join(str(p) for p in paragraphs if str(p).strip())
        for text in chunk_words(section_text, chunk_size, overlap):
            chunks.append(Chunk(f"{record['id']}::{chunk_index}", record['id'], record.get('title', ''), str(section), text))
            chunk_index += 1
    return chunks


def normalise_answer(answer_record):
    data = answer_record.get('answer', answer_record)
    if data.get('unanswerable'):
        return 'Unanswerable'
    if data.get('free_form_answer'):
        return str(data['free_form_answer']).strip()
    spans = data.get('extractive_spans') or []
    spans = [str(span).strip() for span in spans if str(span).strip()]
    if spans:
        return ' ; '.join(spans)
    if data.get('yes_no') is not None:
        return str(data.get('yes_no'))
    return None


def normalise_evidence(answer_record):
    data = answer_record.get('answer', answer_record)
    evidence = data.get('evidence', answer_record.get('evidence', []))
    return [str(item).strip() for item in evidence if str(item).strip()]


def extract_qa_examples(record):
    qas = record.get('qas', {})
    rows = []
    for question, question_id, answers in zip(qas.get('question', []), qas.get('question_id', []), qas.get('answers', [])):
        gold_answers = []
        evidence = []
        for answer in iter_answer_records(answers):
            normalised = normalise_answer(answer)
            if normalised:
                gold_answers.append(normalised)
            evidence.extend(normalise_evidence(answer))
        rows.append(QAExample(record['id'], question_id, record.get('title', ''), question, gold_answers, evidence))
    return rows

In [ ]:
def normalise_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return ' '.join(text.split())


def token_f1(prediction, gold):
    pred_tokens = normalise_text(prediction).split()
    gold_tokens = normalise_text(gold).split()
    if not pred_tokens or not gold_tokens:
        return float(pred_tokens == gold_tokens)
    common = sum((Counter(pred_tokens) & Counter(gold_tokens)).values())
    if common == 0:
        return 0.0
    precision = common / len(pred_tokens)
    recall = common / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def best_f1(prediction, gold_answers):
    return max([token_f1(prediction, gold) for gold in gold_answers], default=0.0)


def token_overlap_recall(candidate, reference):
    candidate_tokens = set(normalise_text(candidate).split())
    reference_tokens = set(normalise_text(reference).split())
    if not reference_tokens:
        return 0.0
    return len(candidate_tokens & reference_tokens) / len(reference_tokens)


def context_recall(contexts, gold_answers, evidence, threshold=0.45):
    refs = evidence or gold_answers
    if not refs:
        return 0.0
    context_text = ' '.join(chunk.text for chunk in contexts)
    return sum(token_overlap_recall(context_text, ref) >= threshold for ref in refs) / len(refs)


def context_precision(contexts, gold_answers, evidence, threshold=0.25):
    refs = evidence or gold_answers
    if not contexts or not refs:
        return 0.0
    relevant = 0
    for chunk in contexts:
        if max(token_overlap_recall(chunk.text, ref) for ref in refs) >= threshold:
            relevant += 1
    return relevant / len(contexts)


def faithfulness(prediction, contexts, threshold=0.35):
    if normalise_text(prediction) == 'unanswerable':
        return 1.0
    claims = [claim.strip() for claim in re.split(r'[.!?;\n]+', prediction) if claim.strip()]
    if not claims:
        return 0.0
    context_text = ' '.join(chunk.text for chunk in contexts)
    return sum(token_overlap_recall(context_text, claim) >= threshold for claim in claims) / len(claims)


def answer_relevancy(prediction, question, gold_answers):
    question_similarity = token_f1(prediction, question)
    if gold_answers:
        return 0.35 * question_similarity + 0.65 * best_f1(prediction, gold_answers)
    return question_similarity


def answer_string_recall(contexts, gold_answers):
    if not gold_answers:
        return 0.0
    context_text = normalise_text(' '.join(chunk.text for chunk in contexts))
    hits = sum(normalise_text(answer) in context_text for answer in gold_answers if normalise_text(answer))
    return hits / len(gold_answers)

In [ ]:
class DenseRetriever:
    def __init__(self, model_name='sentence-transformers/all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name, device='cuda' if torch.cuda.is_available() else 'cpu')
        self.chunks = []
        self.embeddings = None

    def index(self, chunks):
        self.chunks = chunks
        self.embeddings = self.model.encode([chunk.text for chunk in chunks], normalize_embeddings=True, show_progress_bar=False)

    def search(self, query, top_k=5):
        query_embedding = self.model.encode([query], normalize_embeddings=True)[0]
        scores = np.matmul(self.embeddings, query_embedding)
        indices = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[i], float(scores[i])) for i in indices]


class SmallSeq2SeqGenerator:
    def __init__(self, model_name='google/flan-t5-base'):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)

    def answer(self, question, contexts, max_input_tokens=1024, max_new_tokens=96):
        context_text = '\n\n'.join(
            f'[{idx + 1}] Title: {chunk.title}\nSection: {chunk.section}\n{chunk.text}'
            for idx, chunk in enumerate(contexts)
        )
        prompt = (
            'Answer the question using only the provided context. '
            'If the answer is not in the context, answer Unanswerable.\n\n'
            f'Context:\n{context_text}\n\nQuestion: {question}\nAnswer:'
        )
        inputs = self.tokenizer(prompt, return_tensors='pt', truncation=True, max_length=max_input_tokens).to(self.device)
        outputs = self.model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=2)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True).strip()


class BaseRAGPipeline:
    def __init__(self, retriever_model='sentence-transformers/all-MiniLM-L6-v2', generator_model='google/flan-t5-base', chunk_size=180, overlap=40, top_k=5):
        self.retriever = DenseRetriever(retriever_model)
        self.generator = SmallSeq2SeqGenerator(generator_model)
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.top_k = top_k

    def index_document(self, record):
        chunks = build_document_chunks(record, self.chunk_size, self.overlap)
        self.retriever.index(chunks)

    def answer(self, question):
        retrieved = self.retriever.search(question, top_k=self.top_k)
        contexts = [chunk for chunk, _ in retrieved]
        answer = self.generator.answer(question, contexts)
        return {'answer': answer, 'contexts': contexts, 'scores': [score for _, score in retrieved]}

In [ ]:
def run_base_rag(dataset, limit=5, top_k=5, output_dir='outputs'):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    predictions_path = output_dir / 'base_rag_qasper_predictions.jsonl'
    summary_path = output_dir / 'base_rag_qasper_summary.json'

    pipeline = BaseRAGPipeline(top_k=top_k)
    totals = Counter()
    rows = 0

    with predictions_path.open('w', encoding='utf-8') as f:
        for record in tqdm(dataset, desc='Running base RAG'):
            pipeline.index_document(record)
            for example in extract_qa_examples(record):
                result = pipeline.answer(example.question)
                contexts = result['contexts']
                scores = result['scores']
                prediction = result['answer']

                row = {
                    'doc_id': example.doc_id,
                    'question_id': example.question_id,
                    'title': example.title,
                    'question': example.question,
                    'prediction': prediction,
                    'gold_answers': example.gold_answers,
                    'evidence': example.evidence,
                    'token_f1': best_f1(prediction, example.gold_answers),
                    f'answer_string_recall_at_{top_k}': answer_string_recall(contexts, example.gold_answers),
                    'context_precision': context_precision(contexts, example.gold_answers, example.evidence),
                    'context_recall': context_recall(contexts, example.gold_answers, example.evidence),
                    'faithfulness': faithfulness(prediction, contexts),
                    'answer_relevancy': answer_relevancy(prediction, example.question, example.gold_answers),
                    'contexts': [
                        {**asdict(chunk), 'score': score}
                        for chunk, score in zip(contexts, scores)
                    ],
                }
                f.write(json.dumps(row, ensure_ascii=False) + '\n')

                for key in ['token_f1', f'answer_string_recall_at_{top_k}', 'context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']:
                    totals[key] += row[key]
                rows += 1
                if rows >= limit:
                    metrics = {'examples': rows}
                    metrics.update({f'avg_{key}': value / rows for key, value in totals.items()})
                    summary = {'metrics': metrics, 'predictions_path': str(predictions_path)}
                    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
                    return summary

    metrics = {'examples': rows}
    metrics.update({f'avg_{key}': value / rows for key, value in totals.items()})
    summary = {'metrics': metrics, 'predictions_path': str(predictions_path)}
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    return summary


summary = run_base_rag(ds['validation'], limit=5, top_k=5)
summary